<a href="https://colab.research.google.com/github/umang0015/Ai-DataScience/blob/main/Generative%20AI%20/4SepSession_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
!pip -q install "transformers>=4.44" "peft>=0.13" accelerate

In [3]:
import torch
from transformers import AutoModelForCausalLM

MODEL_ID = "stabilityai/stablelm-zephyr-3b" # Placeholder, replace with your desired model ID

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype = torch.float16,
    attn_implementation= 'sdpa',
    device_map='cuda',
)

print("loaded" , MODEL_ID)
print("parameters" , f"{sum(p.numel() for p in model.parameters()):,}")
print("VRAM allocated:" , f"{torch.cuda.memory_allocated() / 1e9:.2f} GB")

config.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 5.59GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/356 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

loaded stabilityai/stablelm-zephyr-3b
parameters 2,795,443,200
VRAM allocated: 5.59 GB


In [4]:
# ask the model about some prompts
from transformers import AutoTokenizer # Import AutoTokenizer

SYSTEM = (
    "You are an IT helpdesk assistant. Reply in exactly three lines.\n"
    "Line 1: the direct answer, one sentence.\n"
    "Line 2: 'Why: ' followed by one sentence.\n"
    "Line 3: 'Next step: ' followed by one sentence.\n"
    "Never use bullet points. Never apologise. No preamble."
)

PROBES = [
    "My laptop will not connect to the office WiFi.",
    "Outlook keeps asking for my password every hour.",
    "I deleted a file from the shared drive by mistake.",
]

tok = AutoTokenizer.from_pretrained(MODEL_ID) # Initialize the tokenizer

# ask the llm these questions and record answers
for question in PROBES:
  messages = [{"role" : "system" , "content" : SYSTEM} ,
              {"role" : "user" , "content" : question}]
  text = tok.apply_chat_template(messages , tokenize=False , add_generation_prompt=True)
  batch = tok(text, return_tensors= 'pt').to(model.device)
  out = model.generate(**batch , max_new_tokens=120 ,do_sample=False)
  reply = tok.decode(out[0][batch['input_ids'].shape[1]:] , skip_special_tokens=True)

  print("Q:" , question)
  print("A:" , reply) # Corrected typo: pritn to print

tokenizer_config.json:   0%|          | 0.00/5.21k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/587 [00:00<?, ?B/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Q: My laptop will not connect to the office WiFi.
A: Why: Check if the WiFi network is available and not hidden. Also, make sure your laptop is within range of the WiFi signal.

Next step: Restart your laptop and try connecting to the WiFi network again. If the issue persists, check the network settings and configuration on your laptop.
Q: Outlook keeps asking for my password every hour.
A: Why: Outlook is configured to prompt you for your password every hour.
Next step: To resolve this, you can check your Outlook settings to change the login interval or disable the password prompt altogether.
Q: I deleted a file from the shared drive by mistake.
A: Why: The file has been deleted from the shared drive.

Next step: Check with your team members if they have a copy of the file or if they can restore it from backups. If the file cannot be recovered, communicate with the team to identify a suitable replacement and update any relevant documentation.


In [5]:
  RAW = [
    ("My laptop battery drains in under two hours.",
     "Request a battery replacement through the hardware desk.\n"
     "Why: A two-hour runtime on a three-year-old machine means the cell has passed its rated charge cycles.\n"
     "Next step: Raise a hardware ticket and attach your asset tag."),
    ("I cannot open the shared finance folder.",
     "Your account is not in the finance access group.\n"
     "Why: That folder uses group-based permissions rather than individual grants.\n"
     "Next step: Ask your manager to approve the access request in the portal."),
    ("The office WiFi keeps disconnecting on my laptop.",
     "Forget the corporate network and rejoin it with your current credentials.\n"
     "Why: A cached profile from the old authentication settings causes repeated drops.\n"
     "Next step: Open network settings, remove the corporate network, and reconnect."),
    ("Outlook asks for my password every hour.",
     "Clear the stored Office credentials in Credential Manager.\n"
     "Why: A stale token keeps failing silent renewal and forces a fresh prompt.\n"
     "Next step: Remove the Office entries in Credential Manager and restart Outlook."),
    ("I deleted a file from the shared drive by mistake.",
     "Restore it yourself from the recycle bin on the share.\n"
     "Why: Deleted items stay recoverable for thirty days before the retention policy purges them.\n"
     "Next step: Right-click the folder, choose restore previous versions, and pick the file."),
    ("My VPN will not connect from home.",
     "Switch the VPN client to the secondary gateway.\n"
     "Why: The primary gateway rejects new sessions once it reaches its connection limit.\n"
     "Next step: Select the alternate profile in the client and reconnect."),
    ("The printer on floor three prints blank pages.",
     "Replace the toner cartridge in that printer.\n"
     "Why: Blank output with no error usually means an empty cartridge rather than a jam.\n"
     "Next step: Log a facilities ticket for a toner swap on the floor three device."),
    ("Teams audio does not work in meetings.",
     "Set the correct output device in Teams audio settings.\n"
     "Why: Teams keeps its own device selection separate from the Windows default.\n"
     "Next step: Open Teams settings, choose Devices, and select your headset."),
    ("My computer takes ten minutes to start up.",
     "Disable the non-essential startup applications.\n"
     "Why: Each application registered at login delays the desktop until it finishes loading.\n"
     "Next step: Open Task Manager, go to Startup, and disable anything not required."),
    ("I received an email asking me to confirm my password.",
     "Do not reply and report the message as phishing.\n"
     "Why: Internal teams never request credentials by email, so this is an impersonation attempt.\n"
     "Next step: Use the Report Phishing button in Outlook and delete the message."),
    ("My second monitor is not detected.",
     "Reseat the display cable at both ends.\n"
     "Why: A partially seated connector passes power but not the display signal.\n"
     "Next step: Unplug and firmly reconnect the cable, then press Windows plus P."),
    ("I cannot install Python on my work laptop.",
     "Request the package through the software portal.\n"
     "Why: Standard accounts cannot install software outside the approved catalogue.\n"
     "Next step: Search for Python in the portal and submit the install request."),
    ("My mailbox is full and I cannot send email.",
     "Archive messages older than one year to free space.\n"
     "Why: Sending is blocked once the mailbox passes its storage quota.\n"
     "Next step: Run the archive policy in Outlook or empty the deleted items folder."),
    ("The docking station stopped charging my laptop.",
     "Test the dock with a different power adapter.\n"
     "Why: Docks fail at the power supply far more often than at the dock itself.\n"
     "Next step: Swap in a spare adapter from the IT desk and confirm charging resumes."),
    ("My authenticator app stopped generating codes.",
     "Re-enrol your device in the authenticator app.\n"
     "Why: Codes stop matching when the phone clock drifts or the enrolment is revoked.\n"
     "Next step: Contact the service desk to reset the enrolment and scan the new code."),
    ("I cannot access the internal wiki from home.",
     "Connect to the VPN before opening the wiki.\n"
     "Why: The wiki is published on the internal network and is not reachable from public addresses.\n"
     "Next step: Start the VPN client, wait for the connected status, then reload the page."),
    ("Excel crashes when I open a large file.",
     "Open the file with add-ins disabled.\n"
     "Why: Third-party add-ins are the most common cause of crashes on large workbooks.\n"
     "Next step: Launch Excel in safe mode and open the file again."),
    ("My screen flickers when I move the laptop.",
     "Have the display cable inspected by the hardware desk.\n"
     "Why: Flicker tied to physical movement points to a loose internal display connector.\n"
     "Next step: Book a hardware appointment and bring the laptop in."),
    ("I forgot my Windows password.",
     "Reset it yourself through the self-service portal.\n"
     "Why: Password resets are automated and do not need a service desk agent.\n"
     "Next step: Open the password portal on your phone and complete the identity check."),
    ("Zoom shows my camera as unavailable.",
     "Close the other application that is holding the camera.\n"
     "Why: Only one application can claim the camera device at a time.\n"
     "Next step: Quit Teams or the browser tab using video, then rejoin the call."),
    ("The company laptop is running very hot.",
     "Have the cooling vents cleaned by the hardware desk.\n"
     "Why: Dust in the intake blocks airflow and forces the fans to run continuously.\n"
     "Next step: Drop the laptop at the IT desk for a vent clean."),
    ("I need access to the production database.",
     "Submit a privileged access request with a business justification.\n"
     "Why: Production access is time-limited and requires approval from the data owner.\n"
     "Next step: Raise the request in the access portal and name your approver."),
    ("Files on the shared drive open as read-only.",
     "Check whether another user has the file open.\n"
     "Why: The share grants write access to one editor at a time to prevent conflicts.\n"
     "Next step: Look at the file status in the share and ask the current editor to close it."),
    ("My phone will not sync work email.",
     "Re-add the account through the company portal.\n"
     "Why: Device compliance expired, so the mail profile was revoked automatically.\n"
     "Next step: Open the company portal, check in the device, and re-add the mail account."),
    ("The website loads slowly only on my machine.",
     "Clear the browser cache and retry.\n"
     "Why: A stale cached asset forces the browser to wait on requests that no longer resolve.\n"
     "Next step: Clear cached files for the last week and reload the page."),
    ("I cannot join the guest network with my personal phone.",
     "Register the device on the guest portal first.\n"
     "Why: The guest network requires a one-time registration before it issues an address.\n"
     "Next step: Connect to the guest network and complete the sign-up page."),
    ("My keyboard types the wrong characters.",
     "Switch the keyboard layout back to your regional setting.\n"
     "Why: A layout change remaps the keys without changing the printed legends.\n"
     "Next step: Press Windows plus space until the correct layout is shown."),
    ("The shared calendar does not show my colleague's availability.",
     "Ask them to grant you free-busy permission.\n"
     "Why: Calendar visibility is granted per person and defaults to private.\n"
     "Next step: Request availability access from your colleague in Outlook."),
    ("I get a certificate warning on an internal site.",
     "Install the internal root certificate on your machine.\n"
     "Why: Internal sites use a private authority that unmanaged browsers do not trust by default.\n"
     "Next step: Run the certificate package from the software portal and reopen the site."),
    ("My laptop will not wake from sleep.",
     "Force a restart by holding the power button.\n"
     "Why: A driver that fails to resume leaves the machine powered but without display output.\n"
     "Next step: Hold power for ten seconds, restart, then install pending driver updates."),
    ("Downloads are blocked in my browser.",
     "Request an exception for the file type you need.\n"
     "Why: The web filter blocks executable and archive downloads by default policy.\n"
     "Next step: Raise a security exception ticket naming the file and its source."),
    ("I cannot share my screen in Teams.",
     "Grant screen recording permission to Teams.\n"
     "Why: The operating system blocks capture until the application is explicitly allowed.\n"
     "Next step: Open privacy settings, enable screen recording for Teams, and restart it."),
    ("My work phone has no signal in the building.",
     "Enable calling over WiFi on the device.\n"
     "Why: The building shielding blocks the carrier signal on lower floors.\n"
     "Next step: Turn on WiFi calling in the phone settings and join the corporate network."),
    ("The scanner will not send documents to my email.",
     "Correct the destination address stored on the scanner.\n"
     "Why: Scanned documents fail silently when the saved address no longer exists.\n"
     "Next step: Update your address in the scanner address book and rescan."),
    ("Software updates keep failing on my laptop.",
     "Free up disk space before retrying the update.\n"
     "Why: Updates need several gigabytes of working space and abort when the disk is nearly full.\n"
     "Next step: Empty the downloads folder and recycle bin, then run the update again."),
    ("I need a licence for the design software.",
     "Request a licence through the software portal.\n"
     "Why: Licences are assigned per user from a fixed pool and cannot be shared.\n"
     "Next step: Submit the licence request and include your cost centre code."),
    ("My files are not backing up to the cloud.",
     "Sign back into the sync client.\n"
     "Why: Sync stops when the authentication token expires after a password change.\n"
     "Next step: Open the sync client, sign in again, and confirm the folder shows as up to date."),
    ("The conference room display shows no signal.",
     "Switch the display to the correct input source.\n"
     "Why: The room display keeps its last input, which is often the previous meeting's device.\n"
     "Next step: Use the room remote to select the input for the table cable."),
    ("I cannot log in after changing my password.",
     "Sign out and back in on every device.\n"
     "Why: Cached credentials on other devices keep submitting the old password and lock the account.\n"
     "Next step: Update the password on your phone and any saved profiles, then retry."),
    ("My laptop fan runs constantly.",
     "Identify the process consuming the processor.\n"
     "Why: Sustained fan noise almost always tracks a single runaway process rather than a fault.\n"
     "Next step: Open Task Manager, sort by CPU, and close the top consumer."),
    ("The team drive is missing from my file explorer.",
     "Remap the network drive with the current path.\n"
     "Why: The share moved to a new server and the old mapping no longer resolves.\n"
     "Next step: Map the drive again using the path listed on the intranet."),
    ("I keep getting meeting invites in the wrong time zone.",
     "Correct the time zone in your calendar settings.\n"
     "Why: Outlook renders every invitation against the profile time zone, not the device clock.\n"
     "Next step: Set the correct time zone in Outlook calendar options."),
    ("My antivirus flagged a file I need.",
     "Submit the file for security review before using it.\n"
     "Why: Detections are sometimes generic matches, but the file stays quarantined until security clears it.\n"
     "Next step: Raise a false positive ticket and attach the detection name."),
    ("The USB headset is not recognised.",
     "Try the headset in a different USB port.\n"
     "Why: Ports on the dock share a controller that can stop enumerating new devices.\n"
     "Next step: Plug the headset directly into the laptop and check the sound settings."),
    ("I cannot open an attachment someone sent.",
     "Ask the sender to share it as a link instead.\n"
     "Why: Attachments above the size limit are stripped before delivery.\n"
     "Next step: Request a shared drive link from the sender."),
    ("My account is locked out.",
     "Unlock it through the self-service portal.\n"
     "Why: Repeated failed sign-ins trigger an automatic thirty-minute lockout.\n"
     "Next step: Complete the identity check in the portal to unlock immediately."),
    ("The remote desktop session disconnects after a few minutes.",
     "Check your local network stability first.\n"
     "Why: Remote sessions drop when packet loss exceeds the client tolerance.\n"
     "Next step: Switch to a wired connection or move closer to the access point."),
    ("I need my laptop encrypted for travel.",
     "Confirm that disk encryption is already active.\n"
     "Why: All managed laptops ship with full disk encryption enabled by policy.\n"
     "Next step: Check the encryption status in system settings and report it if disabled."),
    ("The mouse pointer moves on its own.",
     "Clean the touchpad surface and disable it while typing.\n"
     "Why: Moisture or debris on the touchpad registers as continuous input.\n"
     "Next step: Wipe the touchpad and enable the disable-while-typing option."),
    ("I cannot add a member to my SharePoint site.",
     "Ask the site owner to add them.\n"
     "Why: Only owners can change membership, and you currently hold member rights.\n"
     "Next step: Send the request to the listed site owner."),
    ("My laptop shows a disk almost full warning.",
     "Move large local files to the cloud drive.\n"
     "Why: Local disks fill quickly when downloads and recordings stay on the device.\n"
     "Next step: Move your downloads folder contents to the cloud drive and empty the recycle bin."),
    ("Emails from a supplier go to junk.",
     "Add the supplier domain to your safe senders list.\n"
     "Why: The filter learned from earlier user reports and now scores that domain as spam.\n"
     "Next step: Add the domain in Outlook junk email options and move the messages back."),
    ("I cannot record a Teams meeting.",
     "Ask the organiser to start the recording.\n"
     "Why: Recording rights belong to the organiser and presenters, not to attendees.\n"
     "Next step: Request presenter rights or ask the organiser to record."),
    ("The projector image is stretched.",
     "Set the display resolution to match the projector.\n"
     "Why: A mismatched aspect ratio stretches the image to fill the panel.\n"
     "Next step: Change the resolution in display settings to the projector's native size."),
    ("My browser bookmarks disappeared.",
     "Sign in to the browser profile to restore them.\n"
     "Why: Bookmarks live in the synced profile and vanish when the browser opens as a guest.\n"
     "Next step: Sign in to the browser with your work account and wait for sync."),
    ("I need to wipe a laptop before returning it.",
     "Hand it to the IT desk for a managed wipe.\n"
     "Why: Devices must be wiped through the management console to clear the asset record.\n"
     "Next step: Return the laptop to the IT desk with its charger and asset tag."),
    ("The file I saved on the desktop is gone.",
     "Look in the cloud-synced desktop folder.\n"
     "Why: Desktop folders are redirected to cloud storage, so files move rather than disappear.\n"
     "Next step: Open the cloud drive and check the desktop folder."),
    ("I cannot dial into the conference bridge.",
     "Use the local dial-in number for your region.\n"
     "Why: The default number in the invitation is international and is blocked on some plans.\n"
     "Next step: Open the invitation, expand local numbers, and dial the regional one."),
    ("My laptop asks for a disk recovery key.",
     "Retrieve the recovery key from your account portal.\n"
     "Why: A firmware or hardware change triggers recovery mode to verify the device.\n"
     "Next step: Sign in to the recovery portal on another device and enter the key shown."),
    ("The application freezes when I export a report.",
     "Export the report in smaller date ranges.\n"
     "Why: The export holds the full result set in memory and stalls past a few hundred thousand rows.\n"
     "Next step: Split the export into monthly ranges and combine the files afterwards."),
    ("I cannot see the shared mailbox.",
     "Ask for delegated access to the mailbox.\n"
     "Why: Shared mailboxes appear only after your account is granted full access rights.\n"
     "Next step: Request access from the mailbox owner and restart Outlook."),
    ("My headset microphone is very quiet.",
     "Raise the microphone level in sound settings.\n"
     "Why: New devices default to a low input gain rather than a calibrated one.\n"
     "Next step: Open sound settings, select the headset input, and increase the level."),
    ("The internal portal shows an error after login.",
     "Clear the site cookies and sign in again.\n"
     "Why: An expired session cookie conflicts with the new sign-in and returns an error.\n"
     "Next step: Clear cookies for that site and retry the login."),
    ("I need to send a file that is too large for email.",
     "Share it through the cloud drive instead.\n"
     "Why: Attachments above twenty-five megabytes are rejected by the mail gateway.\n"
     "Next step: Upload the file to your cloud drive and send the share link."),
    ("My laptop clock is wrong.",
     "Enable automatic time synchronisation.\n"
     "Why: A drifting clock breaks authentication tokens as well as calendar entries.\n"
     "Next step: Turn on automatic time and time zone in date settings."),
    ("The wireless mouse stops responding.",
     "Replace the battery in the mouse.\n"
     "Why: Intermittent response is the standard symptom of a battery near the end of its charge.\n"
     "Next step: Fit a fresh battery and re-pair the mouse if needed."),
    ("I cannot access a website that colleagues can reach.",
     "Check whether your machine is using the correct name resolution.\n"
     "Why: A stale cached entry resolves the address to a server that no longer answers.\n"
     "Next step: Flush the resolver cache and reload the site."),
    ("My work account signs me out constantly.",
     "Check the access policy applied to your device.\n"
     "Why: Non-compliant devices are given short sessions and re-prompted frequently.\n"
     "Next step: Confirm device compliance in the company portal."),
    ("The PDF opens in the browser instead of the reader.",
     "Change the default application for PDF files.\n"
     "Why: A browser update reclaimed the file association during install.\n"
     "Next step: Set the PDF reader as default in the default apps settings."),
    ("I need admin rights to change a network setting.",
     "Request temporary elevation for the task.\n"
     "Why: Standard accounts cannot alter adapter settings under the security baseline.\n"
     "Next step: Raise an elevation request describing the change and its duration."),
    ("The team inbox is not receiving external mail.",
     "Check the external sender rules on that mailbox.\n"
     "Why: A transport rule can silently quarantine external mail for shared mailboxes.\n"
     "Next step: Ask the mail team to review the rules applied to that address."),
    ("My laptop screen is dim even at full brightness.",
     "Disable adaptive brightness on the display.\n"
     "Why: The ambient light sensor overrides the manual setting when adaptive brightness is on.\n"
     "Next step: Turn off adaptive brightness in display settings."),
    ("I cannot install updates because the disk is locked.",
     "Restart the machine before running the update.\n"
     "Why: A pending operation holds a lock that only clears on reboot.\n"
     "Next step: Restart the laptop and run the update immediately after login."),
    ("The mobile app rejects my work credentials.",
     "Sign in with your full email address rather than your username.\n"
     "Why: Mobile applications authenticate against the cloud directory, which uses the email format.\n"
     "Next step: Retry the sign-in using your full work email address."),
    ("My calendar invitations are not reaching external guests.",
     "Verify the external address before resending.\n"
     "Why: Invitations to mistyped domains fail silently without a bounce message.\n"
     "Next step: Confirm the address with the guest and send the invitation again."),
    ("The laptop does not recognise the ethernet cable.",
     "Test the cable in a different wall port.\n"
     "Why: Individual desk ports are patched separately and some remain inactive.\n"
     "Next step: Move to a known working port and report the dead one to facilities."),
    ("I want to use a personal laptop for work.",
     "Access work applications through the virtual desktop instead.\n"
     "Why: Personal devices cannot be enrolled, so data must stay inside the managed environment.\n"
     "Next step: Request virtual desktop access through the service portal."),
    ("My password expires tomorrow and I am travelling.",
     "Change the password now while you are still on the network.\n"
     "Why: Changing it remotely without the VPN leaves cached credentials out of step.\n"
     "Next step: Update the password today and sign in again on every device."),
    ("The chat history in Teams is missing.",
     "Check whether you are signed into the correct organisation.\n"
     "Why: Teams keeps separate history per tenant when you belong to more than one.\n"
     "Next step: Switch to your primary organisation in the Teams account menu."),
    ("I cannot print from the mobile app.",
     "Use the follow-me print queue instead of a direct printer.\n"
     "Why: Mobile devices cannot reach printer addresses on the internal print network.\n"
     "Next step: Send the job to the follow-me queue and release it at any device."),
]

print(len(RAW), "examples")

80 examples


In [6]:
#Display one question and answer
# with system and user , assistant format
question ,  answers = RAW[0]
print("System" , SYSTEM);print()
print("Q:" , question)
print("A:" , answers)

System You are an IT helpdesk assistant. Reply in exactly three lines.
Line 1: the direct answer, one sentence.
Line 2: 'Why: ' followed by one sentence.
Line 3: 'Next step: ' followed by one sentence.
Never use bullet points. Never apologise. No preamble.

Q: My laptop battery drains in under two hours.
A: Request a battery replacement through the hardware desk.
Why: A two-hour runtime on a three-year-old machine means the cell has passed its rated charge cycles.
Next step: Raise a hardware ticket and attach your asset tag.


In [7]:
# now split the data into train and test set
import random
random.seed(23)
random.shuffle(RAW)
train = RAW[:int(0.8*len(RAW))]
test = RAW[int(0.8*len(RAW)):]


In [8]:
!pip install --upgrade torchao==0.16.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 66.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [9]:
#configure the lora
 # Upgrade torchao to a compatible version
from peft import LoraConfig, get_peft_model,TaskType
lora_config = LoraConfig(
    r=16,lora_alpha=32,lora_dropout=0.05,
    bias="none",task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj","k_proj","v_proj","o_proj" , "gate_proj" , "up_proj" , "down_proj"]
)

#store it to a object
model= get_peft_model(model,lora_config)
model.print_trainable_parameters() # Corrected method name to plural 'parameters'

trainable params: 25,034,752 || all params: 2,820,477,952 || trainable%: 0.8876


In [10]:
# train the model
from transformers import Trainer , TrainingArguments  , DataCollatorForSeq2Seq
args = TrainingArguments(output_dir="Qwen-IT_Helpsesk", per_device_train_batch_size=2,
                         gradient_accumulation_steps=2, num_train_epochs=5 , learning_rate = 2e-4 , lr_scheduler_type="cosine" ,
                         fp16=True, gradient_checkpointing= True,
                         save_strategy='no' ,report_to = 'none' , logging_steps=5)

In [11]:
# plan the trainer with help of data collator
# data collator passes data in sequence
trainer = Trainer(model=model, args=args,
                  train_dataset=train,
                  data_collator=DataCollatorForSeq2Seq(tok, padding=True))

In [12]:
# preprocess the data,
# label masking : using the chat template format text and also mask the labels acrosss the prompt tokens
# cross-check the masking and then apply pass to adapter trainer

In [14]:
# appky the masking to the data
train_ds =[]
for question , answer in train:
  messages = [{"role" : "system" , "content" : SYSTEM},
              {"role" : "user" , "content": question}]
  text_tokens = tok.apply_chat_template(messages ,tokenize=True, add_generation_prompt=True)
  text_ids = text_tokens['input_ids']
  answer_ids = tok(answer , add_special_tokens = False)["input_ids"]
  train_ds.append({"input_ids" : text_ids+ answer_ids , "labels": [-100]* len(text_ids)+ answer_ids})

In [16]:
# check the masked values
example = train_ds[2]
supervised= [ t for t , label in zip(example['input_ids'] , example["labels"]) if label != 100]
print(tok.decode(supervised))

<|system|>
You are an IT helpdesk assistant. Reply in exactly three lines.
Line 1: the direct answer, one sentence.
Line 2: 'Why: ' followed by one sentence.
Line 3: 'Next step: ' followed by one sentence.
Never use bullet points. Never apologise. No preamble.<|endoftext|>
<|user|>
Teams audio does not work in meetings.<|endoftext|>
<|assistant|>
Set the correct output device in Teams audio settings.
Why: Teams keeps its own device selection separate from the Windows default.
Next step: Open Teams settings, choose Devices, and select your headset.


In [17]:
print(repr (tok.decode(supervised)))

"<|system|>\nYou are an IT helpdesk assistant. Reply in exactly three lines.\nLine 1: the direct answer, one sentence.\nLine 2: 'Why: ' followed by one sentence.\nLine 3: 'Next step: ' followed by one sentence.\nNever use bullet points. Never apologise. No preamble.<|endoftext|>\n<|user|>\nTeams audio does not work in meetings.<|endoftext|>\n<|assistant|>\nSet the correct output device in Teams audio settings.\nWhy: Teams keeps its own device selection separate from the Windows default.\nNext step: Open Teams settings, choose Devices, and select your headset."


In [18]:
# configure the lora

from peft import LoraConfig, get_peft_model, TaskType
lora_config = LoraConfig(
    r=16, lora_alpha=32, target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)
# STORE IT TO OBJECT
model = get_peft_model(model,lora_config)
model.print_trainable_parameters()

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 25,034,752 || all params: 2,820,477,952 || trainable%: 0.8876


In [23]:
# train the model
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
Adaption_dir = "/content/drive/MyDrive/Models"
args = TrainingArguments(
    output_dir=Adaption_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    learning_rate = 2e-4,lr_scheduler_type = "cosine",
    fp16=True,gradient_checkpointing=True,
    save_strategy='no',report_to="none",
    logging_steps=5)

In [24]:
# Plan the trainer with help of data collator
# data collator passes data is sequnce
trainer = Trainer(model = model,
        args = args, train_dataset=train_ds,
        data_collator=DataCollatorForSeq2Seq(tok,padding=True))

In [25]:
# preprocess the data,
# label masking : using the chattemplate format the text and also mask the labels across the prompt tokens
# cross check the making and then pass to adapter trainer
trainer.train()

Step,Training Loss
5,0.060003
10,0.079880
15,0.104821
20,0.094138
25,0.060041
30,0.151915
35,0.057587
40,0.036442
45,0.044626
50,0.026413


TrainOutput(global_step=80, training_loss=0.04855864037526771, metrics={'train_runtime': 179.4008, 'train_samples_per_second': 1.784, 'train_steps_per_second': 0.446, 'total_flos': 732831396790272.0, 'train_loss': 0.04855864037526771, 'epoch': 5.0})

In [26]:
model.save_pretrained(Adaption_dir)
print('saved')

saved


In [27]:
import gc
del trainer
del model
gc.collect()
torch.cuda.empty_cache()
print("Fresh memory allocated" , torch.cuda.memory_allocated())

Fresh memory allocated 17042944


In [28]:
# club the base model with adapter we trained
from peft import PeftModel
base = AutoModelForCausalLM.from_pretrained(MODEL_ID ,
                                            torch_dtype=torch.float16,
                                            attn_implementation='sdpa' ,
                                            device_map='cuda')
# club base plus adapter
Tuned_LLM = PeftModel.from_pretrained(base,Adaption_dir)
Tuned_LLM.eval()
print("base plus adapter loaded")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/356 [00:00<?, ?it/s]

base plus adapter loaded


/usr/local/lib/python3.13/dist-packages/peft/peft_model.py:665: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_B.default.we

In [30]:
# check same IO on tuned llm
for question in PROBES:
  messages = [{"role" : "system" , "content" : SYSTEM} ,
              {"role" : "user" , "content" : question}]
  text = tok.apply_chat_template(messages , tokenize=False , add_generation_prompt=True)
  batch = tok(text, return_tensors= 'pt').to(Tuned_LLM.device)
  out = Tuned_LLM.generate(**batch , max_new_tokens=120 ,do_sample=False)
  reply = tok.decode(out[0][batch['input_ids'].shape[1]:] , skip_special_tokens=True)

  print("Q:" , question)
  print("A:" , reply)

Q: My laptop will not connect to the office WiFi.
A: Why: Check if the WiFi network is available and not hidden. Also, make sure your laptop is within range of the WiFi signal.

Next step: Restart your laptop and try connecting to the WiFi network again. If the issue persists, check the network settings and configuration on your laptop.
Q: Outlook keeps asking for my password every hour.
A: Why: Outlook is configured to prompt you for your password every hour.
Next step: To resolve this, you can check your Outlook settings to change the login interval or disable the password prompt altogether.
Q: I deleted a file from the shared drive by mistake.
A: Why: The file has been deleted from the shared drive.

Next step: Check with your team members if they have a copy of the file or if they can restore it from backups. If the file cannot be recovered, communicate with the team to identify a suitable replacement and update any relevant documentation.
